# PK/PD Hybrid Quantum Model — IBM Backend Evaluation

Run a trained HierarchicalHQCNN model on:
1. **Ideal** simulator (`default.qubit`)
2. **Aer** simulator (`qiskit.aer`)
3. **FakeTorino** noisy simulator
4. **Real IBM hardware**

In [ ]:
import pennylane as qml
import torch
import torch.nn as nn
import numpy as np
import json
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.fake_provider import FakeTorino

# Reuse the folder-name parser and training-time constants from dose_prediction.py
# (importing the module is side-effect-free: argparse only runs inside main())
from dose_prediction import infer_model_config, TIME_WINDOWS, HALF_LIVES
from Utils.data_loader import build_feature_list, prepare_pkpd_data, prepare_lstm_sequences, collate_lstm_batch

## 1. Configuration

In [2]:
# ============================================================
# EDIT THESE
# ============================================================
file_directory = "Results/26_04_22_23_04_54_lstm_dual_stage_h128_combine"
csv_path       = "Data/UpdatedEstData.csv"

plot_patient_ids = [9, 13, 26, 46]
DEVICE = torch.device("cpu")

In [ ]:
# --- Auto-detect model type + feature flags from folder name and checkpoint ---
config = infer_model_config(file_directory)         # {model_type, hidden_dim, mode, combine, stratified}
checkpoint = torch.load(f"{file_directory}/model.pth", map_location="cpu")

mt = config['model_type']

# Decide the data pipeline + eval loop kind from the model family.
if   mt in ('mlp', 'hqcnn', 'qnn'):   data_kind = 'tabular'
elif mt in ('lstm', 'hqlstm'):        data_kind = 'sequence'
elif mt in ('gnn',  'hqgnn'):         data_kind = 'graph'
else: raise ValueError(f"Unsupported model_type: {mt}")

# Infer in_features from the first classical layer of the checkpoint
if data_kind == 'tabular':
    n_features_model = next(v.shape[-1] for k, v in checkpoint.items()
                            if 'weight' in k and v.dim() == 2)
elif data_kind == 'sequence':
    n_features_model = next(v.shape[-1] for k, v in checkpoint.items()
                            if 'input_proj.weight' in k)
else:  # graph
    n_features_model = next(v.shape[-1] for k, v in checkpoint.items()
                            if 'pk_encoder.convs.0' in k and 'weight' in k)

base_n = len(build_feature_list(TIME_WINDOWS, HALF_LIVES))
if   n_features_model == base_n:      add_pk_summary, add_pk_cumulative = False, False
elif n_features_model == base_n + 5:  add_pk_summary, add_pk_cumulative = False, True
elif n_features_model == base_n + 6:  add_pk_summary, add_pk_cumulative = True,  False
elif n_features_model == base_n + 11: add_pk_summary, add_pk_cumulative = True,  True
else:
    raise ValueError(f"Cannot match feature count: model expects {n_features_model}, base={base_n}")

# Number of qubits for the PennyLane backend device
if   mt in ('hqcnn', 'hqlstm', 'hqgnn'): n_qubits_dev = 8  # HQCNN circuit is fixed at 8 wires
elif mt == 'qnn':                         n_qubits_dev = checkpoint['pk_model.q_weights'].shape[1]
else:                                     n_qubits_dev = 4  # classical — q_dev is unused

# Does this model actually use a quantum device? (Determines whether IBM backends matter.)
has_quantum = mt in ('hqcnn', 'qnn', 'hqlstm', 'hqgnn')

print(f"Model:             {mt}  (data_kind={data_kind}, has_quantum={has_quantum})")
print(f"Mode:              {config['mode']}")
print(f"Hidden dim:        {config['hidden_dim']}")
print(f"Combine:           {config['combine']}")
print(f"add_pk_summary:    {add_pk_summary}")
print(f"add_pk_cumulative: {add_pk_cumulative}")
print(f"n_features:        {n_features_model}")
print(f"n_qubits_dev:      {n_qubits_dev}")

if not has_quantum:
    print(f"\n  NOTE: '{mt}' is purely classical — q_dev is ignored; all IBM backends will give identical results.")

## 2. Model Builder

Model type, `mode`, `num_layers`, `n_qubits`, `combine`, and the `add_pk_summary`/`add_pk_cumulative` feature flags are all auto-inferred from the folder name and the checkpoint shape (see cell above). For HQCNN / QNN the PennyLane device `q_dev` is threaded through; for other model types the notebook falls back to `dose_prediction.load_model`.

## 3. Prepare Evaluation Data

In [ ]:
# --- Data prep: dispatch by model family ---
if data_kind == 'tabular':
    data = prepare_pkpd_data(
        csv_path=csv_path,
        test_size=0.1, val_size=0.1, random_state=1712,
        use_perkg=False,
        time_windows=TIME_WINDOWS, half_lives=HALF_LIVES, add_decay=True,
        stratified_split=not config['combine'],
        normalize_data=False,
        add_pk_summary=add_pk_summary,
        add_pk_cumulative=add_pk_cumulative,
    )
    if config['combine']:
        data['val_pk']  = data['train_pk']; data['val_pd']  = data['train_pd']
        data['test_pk'] = data['train_pk']; data['test_pd'] = data['train_pd']

    all_pk = {k: (np.vstack([data['train_pk'][k], data['val_pk'][k], data['test_pk'][k]])
                  if k == 'X' else
                  np.concatenate([data['train_pk'][k], data['val_pk'][k], data['test_pk'][k]]))
              for k in ('X','y','ids','times')}
    all_pd = {k: (np.vstack([data['train_pd'][k], data['val_pd'][k], data['test_pd'][k]])
                  if k == 'X' else
                  np.concatenate([data['train_pd'][k], data['val_pd'][k], data['test_pd'][k]]))
              for k in ('X','y','ids','times')}
    print(f"Tabular: PK samples={len(all_pk['y'])}, PD samples={len(all_pd['y'])}, n_features={data['n_features']}")

elif data_kind == 'sequence':
    data = prepare_lstm_sequences(
        csv_path=csv_path,
        test_size=0.1, val_size=0.1, random_state=1712,
        use_perkg=False,
        time_windows=TIME_WINDOWS, half_lives=HALF_LIVES, add_decay=True,
        stratified_split=not config['combine'],
        combine=config['combine'],
        normalize_data=False,
        add_pk_summary=add_pk_summary,
        add_pk_cumulative=add_pk_cumulative,
    )
    if config['combine']:
        data['val_sequences']  = data['train_sequences']
        data['test_sequences'] = data['train_sequences']
    all_sequences = data['train_sequences'] + data['val_sequences'] + data['test_sequences']
    print(f"Sequence: {len(all_sequences)} patient sequences, n_features={data['n_features']}")

elif data_kind == 'graph':
    from types import SimpleNamespace
    from Utils.data_process import prepare_gnn_data
    gnn_args = SimpleNamespace(
        csv_path='UpdatedEstData',  # prepare_gnn_data adds 'Data/' prefix and '.csv' suffix
        time_windows=TIME_WINDOWS, half_lives=HALF_LIVES,
        add_decay=True, use_perkg=False,
        add_pk_summary=add_pk_summary, add_pk_cumulative=add_pk_cumulative,
        no_placebo=False, stratified_split=not config['combine'],
        test_size=0.1, val_size=0.1, random_state=1712,
        combine=config['combine'], normalize_data=False,
    )
    data = prepare_gnn_data(gnn_args)
    if config['combine']:
        data['val_data']  = data['train_data']
        data['test_data'] = data['train_data']
    all_graphs = data['train_data'] + data['val_data'] + data['test_data']
    print(f"Graph: {len(all_graphs)} graphs, feature_dim={data['feature_dim']}")

## 4. Evaluation Helpers

In [ ]:
@torch.no_grad()
def evaluate_pkpd(model, device, save_path=None):
    """Run the model over all samples. Dispatches by `data_kind` so the same
    function works for tabular / sequence / graph models.
    Returns {metrics, pk: {preds, targets, ids, times}, pd: {...}}."""
    model.eval()

    if data_kind == 'tabular':
        pk_preds = _eval_tabular(model, all_pk['X'], which='pk', device=device)
        pd_preds = _eval_tabular(model, all_pd['X'], which='pd', device=device)
        pk_bundle = {'preds': pk_preds, 'targets': all_pk['y'], 'ids': all_pk['ids'], 'times': all_pk['times']}
        pd_bundle = {'preds': pd_preds, 'targets': all_pd['y'], 'ids': all_pd['ids'], 'times': all_pd['times']}

    elif data_kind == 'sequence':
        pk_bundle, pd_bundle = _eval_sequence(model, all_sequences, device=device)

    else:  # graph
        pk_bundle, pd_bundle = _eval_graph(model, all_graphs, device=device)

    metrics = {
        'pk': _metrics(pk_bundle['targets'], pk_bundle['preds']),
        'pd': _metrics(pd_bundle['targets'], pd_bundle['preds']),
    }

    result = {'metrics': metrics, 'pk': pk_bundle, 'pd': pd_bundle}

    if save_path:
        with open(save_path, 'w') as f:
            json.dump({
                'metrics': metrics,
                'pk_preds':  pk_bundle['preds'].tolist(),
                'pk_targets': pk_bundle['targets'].tolist(),
                'pd_preds':  pd_bundle['preds'].tolist(),
                'pd_targets': pd_bundle['targets'].tolist(),
            }, f, indent=2)
        print(f"  Saved results to {save_path}")

    return result


def _metrics(y_true, y_pred):
    return {
        'MSE':  float(mean_squared_error(y_true, y_pred)),
        'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'MAE':  float(mean_absolute_error(y_true, y_pred)),
        'R2':   float(r2_score(y_true, y_pred)),
    }


def _eval_tabular(model, X, which, device, batch_size=32):
    X_t = torch.as_tensor(X, dtype=torch.float32, device=device)
    preds = []
    n = X_t.shape[0]
    for start in range(0, n, batch_size):
        chunk = X_t[start:start + batch_size]
        if which == 'pk':
            out = model(x_pk=chunk)
            preds.append(out['pk'].cpu().numpy().flatten())
        else:
            out = model(x_pk=chunk, x_pd=chunk)
            preds.append(out['pd'].cpu().numpy().flatten())
    return np.concatenate(preds) if preds else np.zeros(0)


def _eval_sequence(model, sequences, device, batch_size=8):
    pk_preds, pk_targets, pk_ids, pk_times = [], [], [], []
    pd_preds, pd_targets, pd_ids, pd_times = [], [], [], []

    for start in range(0, len(sequences), batch_size):
        batch_seqs = sequences[start:start + batch_size]
        batch = collate_lstm_batch(batch_seqs, device)
        if 'X_pk' not in batch or 'X_pd' not in batch:
            continue

        out = model(
            x_pk=batch['X_pk'], x_pd=batch['X_pd'],
            lengths_pk=batch['lengths_pk'], lengths_pd=batch['lengths_pd'],
        )

        m_pk = batch['mask_pk'].cpu().numpy()
        m_pd = batch['mask_pd'].cpu().numpy()
        pk_p = out['pk'].squeeze(-1).cpu().numpy()
        pd_p = out['pd'].squeeze(-1).cpu().numpy()
        pk_y = batch['y_pk'].cpu().numpy()
        pd_y = batch['y_pd'].cpu().numpy()

        for i, s in enumerate(batch_seqs):
            if 'pk' in s:
                m = m_pk[i][:pk_p.shape[1]]
                pk_preds.append(pk_p[i][m])
                pk_targets.append(pk_y[i][m])
                pk_ids.append(np.full(int(m.sum()), s['id']))
                pk_times.append(s['pk']['times'][:int(m.sum())])
            if 'pd' in s:
                m = m_pd[i][:pd_p.shape[1]]
                pd_preds.append(pd_p[i][m])
                pd_targets.append(pd_y[i][m])
                pd_ids.append(np.full(int(m.sum()), s['id']))
                pd_times.append(s['pd']['times'][:int(m.sum())])

    def _cat(x): return np.concatenate(x) if x else np.zeros(0)
    return (
        {'preds': _cat(pk_preds), 'targets': _cat(pk_targets), 'ids': _cat(pk_ids), 'times': _cat(pk_times)},
        {'preds': _cat(pd_preds), 'targets': _cat(pd_targets), 'ids': _cat(pd_ids), 'times': _cat(pd_times)},
    )


def _eval_graph(model, graphs, device, batch_size=8):
    from torch_geometric.loader import DataLoader as PyGDataLoader
    loader = PyGDataLoader(graphs, batch_size=batch_size, shuffle=False)
    pk_preds, pk_targets, pk_ids, pk_times = [], [], [], []
    pd_preds, pd_targets, pd_ids, pd_times = [], [], [], []
    for batch in loader:
        batch = batch.to(device)
        pd_pred, pk_pred = model(batch, return_pk=True)
        pk_mask = batch.pk_mask.cpu().numpy().astype(bool)
        pd_mask = batch.pd_mask.cpu().numpy().astype(bool)
        pk_preds.append(pk_pred.squeeze(-1).cpu().numpy()[pk_mask])
        pk_targets.append(batch.pk_targets.cpu().numpy()[pk_mask])
        pd_preds.append(pd_pred.squeeze(-1).cpu().numpy()[pd_mask])
        pd_targets.append(batch.pd_targets.cpu().numpy()[pd_mask])
        # best-effort ids/times if the graph carries them
        for attr, sink in (('pk_ids', pk_ids), ('pd_ids', pd_ids),
                          ('pk_times', pk_times), ('pd_times', pd_times)):
            if hasattr(batch, attr):
                vec = getattr(batch, attr).cpu().numpy()
                mask = pk_mask if attr.startswith('pk') else pd_mask
                sink.append(vec[mask])

    def _cat(x): return np.concatenate(x) if x else np.zeros(0)
    return (
        {'preds': _cat(pk_preds), 'targets': _cat(pk_targets), 'ids': _cat(pk_ids), 'times': _cat(pk_times)},
        {'preds': _cat(pd_preds), 'targets': _cat(pd_targets), 'ids': _cat(pd_ids), 'times': _cat(pd_times)},
    )


def print_metrics(name, metrics):
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"  PK - MSE: {metrics['pk']['MSE']:.4f}, RMSE: {metrics['pk']['RMSE']:.4f}, "
          f"MAE: {metrics['pk']['MAE']:.4f}, R2: {metrics['pk']['R2']:.4f}")
    print(f"  PD - MSE: {metrics['pd']['MSE']:.4f}, RMSE: {metrics['pd']['RMSE']:.4f}, "
          f"MAE: {metrics['pd']['MAE']:.4f}, R2: {metrics['pd']['R2']:.4f}")

In [ ]:
def build_model(q_dev):
    """Build the model (type inferred from `config`) and load checkpoint weights.

    Threads `q_dev` through the quantum families (hqcnn/qnn/hqlstm/hqgnn).
    Classical families (mlp/lstm/gnn) ignore q_dev and fall through to
    `dose_prediction.load_model`.
    """
    mt     = config['model_type']
    mode   = config['mode']
    hdim   = config['hidden_dim']
    n_feat = n_features_model

    if mt == 'hqcnn':
        from Models.quantum import HierarchicalHQCNN
        n_layers = sum(1 for k in checkpoint
                       if 'pk_model.qlayers' in k and 'weights_0' in k)
        model = HierarchicalHQCNN(
            pk_input_dim=n_feat, pd_input_dim=n_feat,
            num_layers=max(n_layers, 1), mode=mode, q_dev=q_dev,
        )
        model.load_state_dict(checkpoint)

    elif mt == 'qnn':
        from Models.quantum import HierarchicalQNN
        n_ql, n_qb, _ = checkpoint['pk_model.q_weights'].shape
        model = HierarchicalQNN(
            pk_input_dim=n_feat, pd_input_dim=n_feat,
            n_qubits=n_qb, n_qlayers=n_ql, mode=mode, q_dev=q_dev,
        )
        model.load_state_dict(checkpoint)

    elif mt == 'hqlstm':
        from Models.quantum import HQLSTM
        using_hqcnn = 'pk_encoder.predictor.clayer_1.weight' in checkpoint
        q_w = checkpoint.get('pk_encoder.predictor.q_weights')
        if q_w is not None and not using_hqcnn:
            n_ql, n_qb, _ = q_w.shape
        else:
            n_ql, n_qb = 1, 4
        model = HQLSTM(
            input_dim=n_feat, hidden_dim=hdim, mode=mode,
            n_qlayers=n_ql, n_qubits=n_qb, using_hqcnn=using_hqcnn, q_dev=q_dev,
        )
        model.load_state_dict(checkpoint)

    elif mt == 'hqgnn':
        from Models.quantum import HQGNN
        conv_keys = [k for k in checkpoint if 'pk_encoder.convs.0' in k and 'weight' in k]
        feat_dim  = checkpoint[conv_keys[0]].shape[-1] if conv_keys else n_feat
        using_hqcnn = 'pd_decoder.pd_predictor.clayer_1.weight' in checkpoint
        q_w = checkpoint.get('pd_decoder.pd_predictor.q_weights')
        if q_w is not None and not using_hqcnn:
            n_ql, n_qb, _ = q_w.shape
        else:
            n_ql, n_qb = 1, 4
        model = HQGNN(
            feature_dim=feat_dim, hidden_dim=hdim,
            n_qlayers=n_ql, n_qubits=n_qb, using_hqcnn=using_hqcnn, q_dev=q_dev,
        )
        model.load_state_dict(checkpoint)

    else:
        # mlp / lstm / gnn — purely classical. Reuse dose_prediction.load_model as-is.
        from dose_prediction import load_model
        model = load_model(file_directory, config, n_feat)

    model.eval()
    total = sum(p.numel() for p in model.parameters())
    print(f"  Loaded {mt}: {total:,} parameters")
    return model

## 5. Ideal Simulator (`default.qubit`)

In [7]:
dev_ideal = qml.device("default.qubit", wires=n_qubits_dev)
model_ideal = build_model(dev_ideal)

print("Running Ideal evaluation...")
results_ideal = evaluate_pkpd(
    model_ideal, DEVICE,
    save_path=f"{file_directory}/ibm_ideal_results.json"
)
print_metrics("Ideal (default.qubit)", results_ideal['metrics'])

Loaded LSTM from Results/26_04_22_23_04_54_lstm_dual_stage_h128_combine/model.pth
  Parameters: 1,528,868
  Loaded lstm: 1,528,868 parameters
Running Ideal evaluation...


KeyError: 'pk'

## 6. Aer Simulator

In [ ]:
dev_aer = qml.device("qiskit.aer", wires=n_qubits_dev)
model_aer = build_model(dev_aer)

dev_aer.tracker.active = True
dev_aer.tracker.reset()

print("Running Aer evaluation...")
results_aer = evaluate_pkpd(
    model_aer, DEVICE,
    save_path=f"{file_directory}/ibm_aer_results.json"
)

print_metrics("Aer Simulator", results_aer['metrics'])
print(f"  Executions: {dev_aer.tracker.totals.get('executions', 0)}")
print(f"  Shots: {dev_aer.tracker.totals.get('shots', 0)}")

## 7. FakeTorino (Noisy Simulator)

In [ ]:
backend_fake = FakeTorino()
dev_torino = qml.device("qiskit.remote", wires=n_qubits_dev, backend=backend_fake)
model_torino = build_model(dev_torino)

dev_torino.tracker.active = True
dev_torino.tracker.reset()

print("Running FakeTorino evaluation (noisy simulation)...")
results_torino = evaluate_pkpd(
    model_torino, DEVICE,
    save_path=f"{file_directory}/ibm_torino_results.json"
)

print_metrics("FakeTorino", results_torino['metrics'])
print(f"  Executions: {dev_torino.tracker.totals.get('executions', 0)}")
print(f"  Shots: {dev_torino.tracker.totals.get('shots', 0)}")

## 8. Real IBM Hardware

In [ ]:
# IBM Quantum credentials
json_token = {
    "name": "GIANG_Apr",
    "description": "IBM Quantum API key",
    "createdAt": "2026-04-09T08:58+0000",
    "apikey": "_OO_QDgQz40DKTHRFvITBcJIutsHBj8ygzt2LpINfF8C"
}

QiskitRuntimeService.save_account(
    token=json_token['apikey'],
    name=json_token['name'],
    instance="crn:v1:bluemix:public:quantum-computing:us-east:a/a9b97467e1764d528c81e4fb88381d3e:4d2fdad1-561d-4355-bc80-50f4cd5f5249::",
    overwrite=True,
    channel="ibm_cloud",
)

In [ ]:
service = QiskitRuntimeService()
real_backend = service.least_busy(simulator=False, operational=True)
print(f"Using backend: {real_backend.name}")

dev_real = qml.device("qiskit.remote", wires=n_qubits_dev, backend=real_backend)
model_real = build_model(dev_real)

In [ ]:
dev_real.tracker.active = True
dev_real.tracker.reset()

print(f"Running on IBM Hardware ({real_backend.name})...")
results_real = evaluate_pkpd(
    model_real, DEVICE,
    save_path=f"{file_directory}/ibm_real_results.json"
)

print_metrics(f"Real Hardware ({real_backend.name})", results_real['metrics'])
print(f"  Executions: {dev_real.tracker.totals.get('executions', 0)}")
print(f"  Shots: {dev_real.tracker.totals.get('shots', 0)}")

## 9. Comparison Table

In [ ]:
import pandas as pd

backends = {}
backends["Ideal"] = results_ideal['metrics']
backends["Aer"] = results_aer['metrics']
backends["FakeTorino"] = results_torino['metrics']
if 'results_real' in dir():
    backends[f"Real ({real_backend.name})"] = results_real['metrics']

rows = []
for name, m in backends.items():
    rows.append({
        "Backend": name,
        "PK RMSE": f"{m['pk']['RMSE']:.4f}",
        "PK R2": f"{m['pk']['R2']:.4f}",
        "PD RMSE": f"{m['pd']['RMSE']:.4f}",
        "PD R2": f"{m['pd']['R2']:.4f}",
    })

df_compare = pd.DataFrame(rows)
print(df_compare.to_string(index=False))

# Save comparison
df_compare.to_csv(f"{file_directory}/ibm_comparison.csv", index=False)
print(f"\nSaved to {file_directory}/ibm_comparison.csv")

## 10. Patient Time-Series Plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams.update({"font.family": "serif", "mathtext.fontset": "cm", "font.size": 10})


def plot_patient_comparison(patient_ids, all_results, save_path=None):
    """Plot PK/PD predictions per patient across backends."""
    n_patients = len(patient_ids)
    fig, axes = plt.subplots(n_patients, 2, figsize=(14, 4 * n_patients), squeeze=False)

    colors = {"Ideal": "#2196F3", "Aer": "#4CAF50", "FakeTorino": "#FF9800"}
    if any("Real" in k for k in all_results):
        real_key = [k for k in all_results if "Real" in k][0]
        colors[real_key] = "#F44336"

    for row, pid in enumerate(patient_ids):
        for col, (target_type, title) in enumerate([("pk", "PK"), ("pd", "PD")]):
            ax = axes[row, col]

            # Get ground truth from ideal (same for all)
            r0 = list(all_results.values())[0]
            mask = r0[target_type]['ids'] == pid
            if not mask.any():
                ax.set_title(f"Patient {pid} — {title} (no data)")
                continue

            times = r0[target_type]['times'][mask]
            targets = r0[target_type]['targets'][mask]
            sort_idx = np.argsort(times)
            times = times[sort_idx]
            targets = targets[sort_idx]

            ax.plot(times, targets, 'ko-', label='Ground Truth', markersize=4, linewidth=1.5)

            for name, res in all_results.items():
                preds = res[target_type]['preds'][mask][sort_idx]
                color = colors.get(name, "gray")
                ax.plot(times, preds, '--', color=color, label=name, alpha=0.8, linewidth=1.2)

            ax.set_title(f"Patient {pid} — {title}")
            ax.set_xlabel("Time (h)")
            ax.set_ylabel(f"{title} (DV)")
            if row == 0 and col == 0:
                ax.legend(fontsize=8)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved plot to {save_path}")
    plt.show()

In [ ]:
all_results = {"Ideal": results_ideal, "Aer": results_aer, "FakeTorino": results_torino}
if 'results_real' in dir():
    all_results[f"Real ({real_backend.name})"] = results_real

plot_patient_comparison(
    plot_patient_ids, all_results,
    save_path=f"{file_directory}/ibm_patient_comparison.png"
)

## 11. Circuit Info

In [ ]:
# Draw the QCNN circuit (only meaningful for the hqcnn model type)
if config['model_type'] == 'hqcnn':
    from Models.quantum import _hqcnn_circuit_fn, _HQCNN_WEIGHT_SHAPES, _HQCNN_N_QUBITS
    dummy_dev   = qml.device("default.qubit", wires=_HQCNN_N_QUBITS)
    dummy_qnode = qml.QNode(_hqcnn_circuit_fn, dummy_dev)
    dummy_inputs  = torch.zeros(_HQCNN_N_QUBITS)
    dummy_weights = {k: torch.zeros(v) for k, v in _HQCNN_WEIGHT_SHAPES.items()}

    print("QCNN Circuit:")
    print(qml.draw(dummy_qnode)(dummy_inputs, **dummy_weights))

    specs = qml.specs(dummy_qnode)(dummy_inputs, **dummy_weights)
    print(f"\nGate count: {specs.get('num_used_wires', _HQCNN_N_QUBITS)} qubits")
    print(f"Depth: {specs.get('depth', 'N/A')}")
    print(f"Gate types: {specs.get('gate_types', {})}")
else:
    print(f"Circuit diagram only available for 'hqcnn'; model_type='{config['model_type']}' skipped.")